# 2d Wave equation with hyperboloidal compactification in second order form

In [1]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw



Rout = 2
domain = Circle((0, 0), Rout).Face()
domain.edges.name = 'outer'
domain.faces.name = 'outer'


RScat = 0.3


R=1.
inner = Circle((0, 0), R).Face()
inner.edges.name = 'interface'
inner.faces.name = 'inner'


if RScat == 0:
    Draw(Glue([domain-inner,inner]))
    geo = OCCGeometry(Glue([domain-inner,inner]), dim=2)

else:
    scatterer = Circle((0,0),RScat).Face()
    scatterer.edges.name = 'scatterer'


if RScat<R and RScat >0:
    Draw(Glue([domain-inner,inner-scatterer]))
    geo = OCCGeometry(Glue([domain-inner,inner-scatterer]), dim=2)
else:
    Draw(domain-scatterer)
    geo = OCCGeometry(domain-scatterer, dim=2)



WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

In [2]:
mesh = Mesh(geo.GenerateMesh(maxh=0.06))
order = 2
mesh.Curve(order)
Draw(mesh)
print(mesh.GetMaterials())

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

('outer', 'inner')


In [3]:
def CFGrad3(cf):
    return CF((cf.Diff(x),cf.Diff(y),cf.Diff(z)),dims=(3,cf.shape[0]))
def CFGrad2(cf):
    return CF((cf.Diff(x),cf.Diff(y)),dims=(2,cf.shape[0]))

def Jacobian3(cf):
    return CFGrad3(cf).trans
def Jacobian2(cf):
    return CFGrad2(cf).trans

In [4]:
rho = sqrt(x**2+y**2)

r = R+tan(rho-R)
J = Jacobian2(r/rho*CF((x,y),dims=(2,1)))


h = CF(r+1/(1+r),dims=(1,1))
H = Inv(J.trans)*CFGrad2(h)

w = CF(R/sqrt(r),dims=(1,1))
W = CFGrad2(w)

In [5]:
fes = H1(mesh,order=order)
u,u_ = fes.TnT()

n = specialcf.normal(2)

M_ = (
    u*u_*dx('inner')
    +u*u_*(1-H.trans*H)*w**2*Det(J)*dx('outer')
    )

C_ = (
    H.trans*Inv(J).trans*grad(u_)*u*w**2*Det(J)*dx('outer')
    -H.trans*Inv(J).trans*grad(u)*u_*w**2*Det(J)*dx('outer')
    -H.trans*Inv(J).trans*n*u_*u*w**2*Det(J)*ds('outer')
    )
K_ = (
    -grad(u)*grad(u_)*dx('inner')
    -(Inv(J).trans*grad(u))*(Inv(J).trans*grad(u_))*w**2*Det(J)*dx('outer')
    -InnerProduct((Inv(J).trans*W),(Inv(J).trans*grad(u_)))*u*w*Det(J)*dx('outer')
    -InnerProduct(Inv(J).trans*grad(u),Inv(J).trans*W)*u_*w*Det(J)*dx('outer')
    -InnerProduct(Inv(J).trans*W,Inv(J).trans*W)*u_*u*Det(J)*dx('outer')
    )

## Newmark beta time-stepping

as a first time discretization we use a Newmark $\beta$ scheme:

In [6]:
dt = 1e-2

beta = 1/4
gamma = 1/2

Sinv = BilinearForm(M_-gamma*dt*C_-beta*dt**2*K_).Assemble().mat.Inverse()
K = BilinearForm(-1.0*K_).Assemble().mat
C = BilinearForm(-1.0*C_).Assemble().mat
Minv = BilinearForm(M_).Assemble().mat.Inverse()

used dof inconsistency
(silence this warning by setting BilinearForm(...check_unused=False) )


In [7]:
gfw = GridFunction(fes)
gfw_ = GridFunction(fes)
gfw__ = GridFunction(fes)


# for immediate drawing
#scene = Draw(gfw_,min=0,max=0.1)

# for animation

gfanim = GridFunction(fes,multidim = 0)

In [8]:
w0 = exp(-((x-0.4)**2+(y-0.)**2)*30)

gfw.Set(0.)
gfw_.Set(w0)
gfw__.Set(0.)

gfw__.vec.data = -Minv@K*gfw.vec-Minv@C*gfw_.vec

tmp = gfw__.vec.CreateVector()




tend = 3
drawevery = (.1/dt)
gfanim.AddMultiDimComponent(gfw_.vec)


t = 0.
i = 0

#newmark beta
while t<tend:
    t+=dt
    i+=1
    
    tmp.data = gfw__.vec
    gfw__.vec.data = -Sinv @ K*gfw.vec-Sinv@C*gfw_.vec-dt*Sinv@K*gfw_.vec+(gamma-1)*dt*Sinv@C*tmp+(beta-1/2)*dt**2*Sinv@K*tmp
    gfw.vec.data += dt*gfw_.vec+(1/2-beta)*dt**2*tmp+beta*dt**2*gfw__.vec
    gfw_.vec.data += (1-gamma)*dt*tmp+gamma*dt*gfw__.vec

    
    if (i%drawevery) == 0:
        #print('\r t = {}'.format(t),end="")
        #scene.Redraw()
        gfanim.AddMultiDimComponent(gfw_.vec)

In [9]:
sceneanim = Draw(gfanim,animate=True,min=0,max=0.1)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…